# Approach 4: TAPEX — Pre-training via SQL Execution

**TAPEX** (Table Pre-training via Execution) takes a fundamentally different approach than TaPas.

**Key innovation:** Instead of designing special embeddings, TAPEX **pre-trains a BART model** by teaching it to *execute* synthetic SQL queries. The model sees millions of `(table, SQL_query) → result` examples and learns table understanding as a side effect.

**Model:** `microsoft/tapex-large-finetuned-wtq` (406M parameters, BART-large backbone)  
**Output:** Free-form text (generative) — more flexible than TaPas's cell selection

**TaPas vs TAPEX:**
| | TaPas | TAPEX |
|---|---|---|
| Backbone | BERT (encoder-only) | BART (encoder-decoder) |
| Output | Cell coordinates + aggregation | Free text |
| Pre-training | MLM + table position embeddings | SQL execution simulation |
| WTQ accuracy | ~57% | ~57-60% (larger model) |

In [ ]:
# !pip install transformers torch pandas

In [ ]:
import pandas as pd
from transformers import TapexTokenizer, BartForConditionalGeneration
import torch
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/employees.csv')
df.head()

## Step 1: Load TAPEX

In [ ]:
MODEL_ID = "microsoft/tapex-large-finetuned-wtq"

print(f"Loading {MODEL_ID}...")
tokenizer = TapexTokenizer.from_pretrained(MODEL_ID)
model = BartForConditionalGeneration.from_pretrained(MODEL_ID)
model.eval()
print("Model loaded!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 2: Understand TAPEX's input encoding

TAPEX linearizes the table as a flat sequence:
```
col: name | department | salary ... row 1: Alice Johnson | Engineering | 95000 ... row 2: Bob Smith | ...
```
Then this sequence goes through BART's encoder, and the decoder generates the answer autoregressively.

In [ ]:
table = df.astype(str)
question = "Who has the highest salary?"

encoding = tokenizer(
    table=table.head(5),
    query=question,
    return_tensors='pt',
    truncation=True
)

# Decode to see exactly what the model receives as text
decoded_input = tokenizer.decode(encoding['input_ids'][0], skip_special_tokens=False)
print("Model input (linearized table + question):")
print(decoded_input[:800])
print("...")

## Step 3: Inference function

In [ ]:
def ask_tapex(question: str, table: pd.DataFrame) -> str:
    """Ask TAPEX a question about a table. Returns a free-text answer."""
    table_str = table.astype(str)
    
    encoding = tokenizer(
        table=table_str,
        query=question,
        return_tensors='pt',
        truncation=True,
        max_length=1024
    )
    
    with torch.no_grad():
        output_ids = model.generate(
            **encoding,
            max_length=64,
            num_beams=5,
            early_stopping=True
        )
    
    answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return answer

## Step 4: Q&A demos

In [ ]:
questions = [
    "Who has the highest salary?",
    "How many employees are in Engineering?",
    "What is the average salary?",
    "Which employees have more than 8 years of experience?",
    "Who has the lowest performance score?",
]

for q in questions:
    answer = ask_tapex(q, df)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print()

## Step 5: Multi-hop reasoning — where TAPEX shines over TaPas

In [ ]:
# TAPEX can handle more complex compositional questions
# because it generates free text rather than selecting cells

complex_questions = [
    "Among promoted employees, who has the highest salary?",
    "What is the difference in salary between the highest and lowest earner?",
    "How many employees in Engineering have a performance score above 4.5?",
]

for q in complex_questions:
    answer = ask_tapex(q, df)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print()

## Step 6: Inspect TAPEX's BART backbone

In [ ]:
print("TAPEX Model (BART-Large backbone):")
print(f"  Encoder layers: {model.config.encoder_layers}")
print(f"  Decoder layers: {model.config.decoder_layers}")
print(f"  Hidden size:    {model.config.d_model}")
print(f"  Attention heads:{model.config.encoder_attention_heads}")
print(f"  Max position:   {model.config.max_position_embeddings}")
print(f"  Vocab size:     {model.config.vocab_size:,}")
print()
print("Difference from TaPas:")
print("  - BART has BOTH encoder AND decoder (seq2seq)")
print("  - Generates tokens one at a time (autoregressive)")
print("  - No special row/column embeddings — learns table structure implicitly")
print("    through SQL execution pre-training")

## Step 7: How TAPEX was pre-trained

In [ ]:
# TAPEX pre-training example:
# The model was trained on millions of (table, SQL, result) triples
# like a synthetic SQL executor:

print("TAPEX Pre-training Examples:")
print("=" * 60)
examples = [
    {
        "sql": "SELECT name FROM employees WHERE salary = MAX(salary)",
        "expected_output": "Liam Harris"
    },
    {
        "sql": "SELECT COUNT(*) FROM employees WHERE department = 'Engineering'",
        "expected_output": "7"
    },
    {
        "sql": "SELECT AVG(performance_score) FROM employees",
        "expected_output": "4.14"
    },
]

print("During pre-training, TAPEX saw millions of:")
print("  Input:  [TABLE] + [SQL_QUERY]")
print("  Output: [EXECUTION_RESULT]")
print()
for ex in examples:
    print(f"  SQL: {ex['sql']}")
    print(f"  → Result: {ex['expected_output']}")
    print()

print("This teaches the model to understand table structure")
print("without ever explicitly defining row/column embeddings.")

## Key Takeaways

| | TAPEX |
|---|---|
| **Architecture** | BART encoder-decoder (generative) |
| **Pre-training** | SQL execution simulation |
| **Output** | Free-form text — handles complex answers |
| **Parameters** | 406M (large) |
| **Best for** | Complex table Q&A, compositional reasoning |

**Next:** See `05_tabtransformer.ipynb` — a purpose-built architecture for tabular *classification/regression* (not Q&A).